# 05 - Multimodal Fusion & Cross-Modal Attention
## MedAgent: Multimodal Clinical Decision Support System

This notebook demonstrates multimodal patient representation learning:
1. Pairing ECG biosignals, chest radiographs, and clinical demographics
2. Ingesting into **Cross-Modal Attention Transformer ($9.6	ext{M}$ params)**
3. Extracting and visualizing **Inter-Modality Attention Heatmaps**
4. Multimodal clinical risk stratification


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from src.config.paths import CHECKPOINTS_DIR
from src.data.loaders.image_loader import generate_synthetic_xray_image, get_xray_transforms
from src.data.loaders.multimodal_loader import RISK_LEVELS, determine_clinical_risk
from src.models.fusion.multimodal_transformer import MultimodalTransformer


### 1. Load Multimodal Transformer Checkpoint


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = MultimodalTransformer(
    num_risk_classes=3,
    num_signal_classes=5,
    num_image_classes=5,
    embed_dim=256,
    num_heads=8,
    num_layers=3,
    pretrained_vision=False
).to(device)

ckpt_path = CHECKPOINTS_DIR / "multimodal_transformer_best.pth"
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    print("Loaded trained Multimodal Transformer checkpoint!")
model.eval()


### 2. Construct Multimodal Patient Ingestion


In [ ]:
# 1. ECG waveform tensor: (1, 1, 260)
t = np.linspace(0, 1, 260)
dummy_ecg = np.sin(2 * np.pi * 5 * t).astype(np.float32)
ecg_tensor = torch.from_numpy(dummy_ecg).unsqueeze(0).unsqueeze(0).to(device)

# 2. Image tensor: (1, 3, 224, 224)
img = generate_synthetic_xray_image(pathology="Pleural Effusion")
img_tensor = get_xray_transforms(224, is_training=False)(img).unsqueeze(0).to(device)

# 3. Tabular vector: (1, 8) [Age=72, Sex=Male, SBP=158, DBP=94, HR=98, SpO2=90%, HTN=1, DM=1]
tab_vec = np.array([(72-55)/20, 1.0, (158-120)/20, (94-80)/10, (98-75)/15, (90-95)/5, 1.0, 1.0], dtype=np.float32)
tab_tensor = torch.from_numpy(tab_vec).unsqueeze(0).to(device)

with torch.no_grad():
    out = model(ecg_tensor, img_tensor, tab_tensor)
    risk_probs = torch.softmax(out["risk_logits"], dim=1).squeeze().cpu().numpy()
    patient_emb = out["patient_embedding"].squeeze().cpu().numpy()
    attn_matrix = out["attention_weights"][0].cpu().numpy()

risk_idx = int(np.argmax(risk_probs))
print(f"Stratified Risk Level: {RISK_LEVELS[risk_idx]} ({risk_probs[risk_idx]*100:.1f}% probability)")
print(f"Extracted Unified Patient Embedding Vector Dimension: {patient_emb.shape}")


### 3. Visualizing Cross-Modal Attention Weights
Reveals how the model balances ECG signals, chest imaging, and patient vitals to determine clinical risk.


In [ ]:
modality_labels = ["[Summary]", "ECG Signal", "Chest X-Ray", "Demographics"]

plt.figure(figsize=(7, 6))
sns.heatmap(
    attn_matrix,
    annot=True,
    fmt=".3f",
    cmap="Blues",
    xticklabels=modality_labels,
    yticklabels=modality_labels,
    cbar=True
)
plt.title("Cross-Modal Transformer Attention Matrix", fontsize=13, fontweight="bold")
plt.xlabel("Key Modality Tokens", fontsize=11)
plt.ylabel("Query Modality Tokens", fontsize=11)
plt.tight_layout()
plt.show()
